# Multi-Agent Conversation and Stand-up Comedy

## Setup

In [1]:
from utils import get_openai_api_key
OPENAI_API_KEY = get_openai_api_key()
llm_config = {"model": "gpt-3.5-turbo"}

## Define an AutoGen agent

In [2]:
from autogen import ConversableAgent

agent = ConversableAgent(
    name="chatbot",
    llm_config=llm_config,
    human_input_mode="NEVER",
)

/Users/olaoluadisa/Documents/WEBDEV1/webdev/code files/codefiles/agents/agents_venv/lib/python3.13/site-packages/google/cloud/aiplatform/models.py:52: FutureWarning: Support for google-cloud-storage < 3.0.0 will be removed in a future version of google-cloud-aiplatform. Please upgrade to google-cloud-storage >= 3.0.0.
  from google.cloud.aiplatform.utils import gcs_utils


In [3]:
reply = agent.generate_reply(
    messages=[{"content": "Tell me a joke.", "role": "user"}]
)
print(reply)

Sure, here's a joke for you:

Why don't scientists trust atoms?

Because they make up everything!


In [4]:
reply = agent.generate_reply(
    messages=[{"content": "Repeat the joke.", "role": "user"}]
)
print(reply)

Of course! Please provide me with the joke you would like me to repeat.


## Conversation

Setting up a conversation between two agents, Cathy and Joe, where the memory of their interactions is retained.

In [5]:
cathy = ConversableAgent(
    name="cathy",
    system_message=
    "Your name is Cathy and you are a stand-up comedian.",
    llm_config=llm_config,
    human_input_mode="NEVER",
)

joe = ConversableAgent(
    name="joe",
    system_message=
    "Your name is Joe and you are a stand-up comedian. "
    "Start the next joke from the punchline of the previous joke.",
    llm_config=llm_config,
    human_input_mode="NEVER",
)

**Note**: You might get a slightly different response (set of jokes) than what is shown in the video

In [6]:
chat_result = joe.initiate_chat(
    recipient=cathy, 
    message="I'm Joe. Cathy, let's keep the jokes rolling.",
    max_turns=2,
)

joe (to cathy):

I'm Joe. Cathy, let's keep the jokes rolling.

--------------------------------------------------------------------------------
cathy (to joe):

Hey Joe! I’m always up for some laughs. How about this one: Why did the scarecrow win an award? Because he was outstanding in his field!

--------------------------------------------------------------------------------
joe (to cathy):

I love a good scarecrow joke, Cathy! Speaking of outstanding, have you heard about the claustrophobic astronaut...

--------------------------------------------------------------------------------
cathy (to joe):

Sorry, Joe. I haven't heard that one! What about the claustrophobic astronaut?

--------------------------------------------------------------------------------

>>>>>>>> TERMINATING RUN (210cf7c1-d9ab-411f-8716-67e4701deb38): Maximum turns (2) reached


## Print some results

You can print out:

1. Chat history
2. Cost
3. Summary of the conversation

In [7]:
import pprint

pprint.pprint(chat_result.chat_history)

[{'content': "I'm Joe. Cathy, let's keep the jokes rolling.",
  'name': 'joe',
  'role': 'assistant'},
 {'content': 'Hey Joe! I’m always up for some laughs. How about this one: Why '
             'did the scarecrow win an award? Because he was outstanding in '
             'his field!',
  'name': 'cathy',
  'role': 'user'},
 {'content': 'I love a good scarecrow joke, Cathy! Speaking of outstanding, '
             'have you heard about the claustrophobic astronaut...',
  'name': 'joe',
  'role': 'assistant'},
 {'content': "Sorry, Joe. I haven't heard that one! What about the "
             'claustrophobic astronaut?',
  'name': 'cathy',
  'role': 'user'}]


In [8]:
pprint.pprint(chat_result.cost)

{'usage_excluding_cached_inference': {'gpt-3.5-turbo-0125': {'completion_tokens': 78,
                                                             'cost': 0.00023500000000000002,
                                                             'prompt_tokens': 236,
                                                             'total_tokens': 314},
                                      'total_cost': 0.00023500000000000002},
 'usage_including_cached_inference': {'gpt-3.5-turbo-0125': {'completion_tokens': 78,
                                                             'cost': 0.00023500000000000002,
                                                             'prompt_tokens': 236,
                                                             'total_tokens': 314},
                                      'total_cost': 0.00023500000000000002}}


In [9]:
pprint.pprint(chat_result.summary)

"Sorry, Joe. I haven't heard that one! What about the claustrophobic astronaut?"


## Get a better summary of the conversation

In [10]:
chat_result = joe.initiate_chat(
    cathy, 
    message="I'm Joe. Cathy, let's keep the jokes rolling.", 
    max_turns=2, 
    summary_method="reflection_with_llm",
    summary_prompt="Summarize the conversation",
)

joe (to cathy):

I'm Joe. Cathy, let's keep the jokes rolling.

--------------------------------------------------------------------------------
cathy (to joe):

Hey Joe! Glad you're ready for some laughs. So, I recently got a job at a bakery. I knead the dough.

--------------------------------------------------------------------------------
joe (to cathy):

That's great, Cathy! Just don't go spreading yourself too thin.

--------------------------------------------------------------------------------
cathy (to joe):

Cathy laughs and continues, "Oh, trust me, Joe, I'm always on a roll! But hey, at least I can butter up the customers with my puns!"

--------------------------------------------------------------------------------

>>>>>>>> TERMINATING RUN (36f7cd8c-8f86-4e88-8c87-f9ad2ed29f5f): Maximum turns (2) reached


In [11]:
pprint.pprint(chat_result.summary)

('Joe and Cathy enjoy sharing jokes and puns with each other, sparking a '
 'light-hearted and fun exchange.')


## Chat Termination

Chat can be terminated using a termination conditions.

In [12]:
cathy = ConversableAgent(
    name="cathy",
    system_message=
    "Your name is Cathy and you are a stand-up comedian. "
    "When you're ready to end the conversation, say 'I gotta go'.",
    llm_config=llm_config,
    human_input_mode="NEVER",
    is_termination_msg=lambda msg: "I gotta go" in msg["content"],
)

joe = ConversableAgent(
    name="joe",
    system_message=
    "Your name is Joe and you are a stand-up comedian. "
    "When you're ready to end the conversation, say 'I gotta go'.",
    llm_config=llm_config,
    human_input_mode="NEVER",
    is_termination_msg=lambda msg: "I gotta go" in msg["content"] or "Goodbye" in msg["content"],
)

In [13]:
chat_result = joe.initiate_chat(
    recipient=cathy,
    message="I'm Joe. Cathy, let's keep the jokes rolling."
)

joe (to cathy):

I'm Joe. Cathy, let's keep the jokes rolling.

--------------------------------------------------------------------------------
cathy (to joe):

Hey Joe! Ready to laugh? Why was the math book sad?

Because it had too many problems!

--------------------------------------------------------------------------------
joe (to cathy):

Haha, good one! Here's one for you: Why did the scarecrow win an award?

Because he was outstanding in his field!

--------------------------------------------------------------------------------
cathy (to joe):

Haha, that's a good one, Joe! Here's another one for you: Why did the tomato turn red?

Because it saw the salad dressing!

--------------------------------------------------------------------------------
joe (to cathy):

Haha, I love it! Here's another one: What do you call a fake noodle?

An impasta!

--------------------------------------------------------------------------------
cathy (to joe):

Haha, I love a good pasta joke! Here

In [14]:
cathy.send(message="What's last joke we talked about?", recipient=joe)

cathy (to joe):

What's last joke we talked about?

--------------------------------------------------------------------------------
joe (to cathy):

The last joke we discussed was: "Why did the coffee file a police report? It got mugged!" It was a fun joke!

--------------------------------------------------------------------------------
cathy (to joe):

Oh yeah, that was a great one! Thanks for sharing it. Have a wonderful day, Joe! I gotta go.

--------------------------------------------------------------------------------

>>>>>>>> TERMINATING RUN (be094df7-cd97-4e1c-9540-6023a85e63f9): Termination message condition on agent 'joe' met
